# Giai đoạn 3 — Bước 8B: FastAPI Local API Client

**Bạn là người trực tiếp viết `backend/app/main.py`, chạy Uvicorn ở PowerShell và chạy notebook này.**

- 8A đã xong; KHÔNG `.fit()`, KHÔNG Test, KHÔNG sửa SHA256/ngưỡng 0,225.
- Chỉ sử dụng dữ liệu **giả lập**. Không gửi thông tin sức khỏe cá thể.
- **Trước khi chạy code ô bên dưới**, đọc `README-Buoc-8B-FastAPI.md` và khởi động server cục bộ tại `http://127.0.0.1:8000`.
- Notebook này gửi HTTP request; không tự khởi động server và không nạp model.

## 1. Chuẩn bị đường dẫn và thư viện

Yêu cầu: `requests` đã cài trong **cùng kernel Jupyter**. Nếu chưa có, chạy `%pip install requests` trong ô mới do bạn tạo. Không cập nhật scikit-learn.

In [5]:
from pathlib import Path
from datetime import datetime, timezone
import json
import requests
import platform

API_ROOT = 'http://127.0.0.1:8000'
PREDICT_URL = API_ROOT + '/api/v1/research/nhanes/hba1c/predict'
SHA = '31e9c94b0f14369be7920b9315b026d0c4afedb9f7b2885d120d631d930582b1'
THRESHOLD = 0.225
EXPECTED_SCORES = [0.22552, 0.39459]  # Chỉ cho hai ví dụ tổng hợp Bước 8A

cwd = Path.cwd().resolve()
if (cwd / 'data').is_dir():
    ROOT = cwd
elif (cwd.parent / 'data').is_dir():
    ROOT = cwd.parent
else:
    raise RuntimeError('Mở notebook từ 03-ML-Research hoặc thư mục notebooks bên trong.')
OUT = ROOT / 'data' / 'outputs' / 'model_deployment' / 'fastapi_8b'
print('Notebook environment:', platform.python_version())
print('API:', API_ROOT)
print('CHỈ synthetic tests | FIT: NO | TEST: NO')

Notebook environment: 3.13.9
API: http://127.0.0.1:8000
CHỈ synthetic tests | FIT: NO | TEST: NO


## 2. Kiểm tra Health API

Nếu ô này báo `ConnectionError`, server chưa chạy hoặc không đúng port. Chuyển sang PowerShell chứa Uvicorn kiểm tra lỗi, không chạy huấn luyện lại.

In [9]:
live = requests.get(API_ROOT + '/health/live', timeout=5)
ready = requests.get(API_ROOT + '/health/ready', timeout=5)
print('live:', live.status_code, live.text)
print('ready:', ready.status_code, ready.text)
assert live.status_code == 200
assert ready.status_code == 200
assert ready.json()['model_sha256'] == SHA
assert ready.json()['research_only'] is True

live: 200 {"status":"alive"}
ready: 200 {"status":"ready","model_sha256":"31e9c94b0f14369be7920b9315b026d0c4afedb9f7b2885d120d631d930582b1","research_only":true}


## 3. Gửi 2 ca dữ liệu giả lập (không gửi hồ sơ người thật)

Đối chiếu điểm với Bước 8A. Sai lệch số học dưới `0.001` được phép. Tuyệt đối không thay ngưỡng hoặc model để khớp test.

In [10]:
synthetic_cases = [
    {'RIDAGEYR': 45, 'RIAGENDR': 1, 'BMXBMI': 28.5},
    {'RIDAGEYR': 60, 'RIAGENDR': 2, 'BMXBMI': 34.8},
]
valid_results = []
for idx, (payload, expected_score) in enumerate(zip(synthetic_cases, EXPECTED_SCORES), 1):
    response = requests.post(PREDICT_URL, json=payload, timeout=10)
    assert response.status_code == 200, f'Case {idx}: {response.status_code} {response.text}'
    data = response.json()
    assert data['status'] == 'research_only'
    assert data['task'] == 'classify_hba1c_ge_5_7_indicator'
    assert data['model_sha256'] == SHA
    assert data['threshold'] == THRESHOLD
    assert data['predicted_label'] == int(data['model_score'] >= THRESHOLD)
    assert 'diabetes_probability' not in data
    assert 'not a diagnosis' in data['warning']
    assert abs(data['model_score'] - expected_score) < 0.001, f'Case {idx}: điểm khác Bước 8A'
    valid_results.append({'case': idx, 'status_code': response.status_code,
                          'score': round(data['model_score'], 5), 'label': data['predicted_label']})
print('TWO SYNTHETIC CASES OK:')
for row in valid_results:
    print(row)

TWO SYNTHETIC CASES OK:
{'case': 1, 'status_code': 200, 'score': 0.22552, 'label': 1}
{'case': 2, 'status_code': 200, 'score': 0.39459, 'label': 1}


## 4. Kiểm tra 6 dữ liệu không hợp lệ

FastAPI/Pydantic mặc định trả HTTP 422 cho request không hợp lệ. Không nạp Test, không truyền thông tin thật.

In [11]:
invalid_cases = [
    ('age_out_of_range', {'RIDAGEYR': 19, 'RIAGENDR': 1, 'BMXBMI': 28.5}),
    ('invalid_gender', {'RIDAGEYR': 45, 'RIAGENDR': 3, 'BMXBMI': 28.5}),
    ('bmi_out_of_range', {'RIDAGEYR': 45, 'RIAGENDR': 1, 'BMXBMI': 100.0}),
    ('missing_bmi', {'RIDAGEYR': 45, 'RIAGENDR': 1}),
    ('extra_field', {'RIDAGEYR': 45, 'RIAGENDR': 1, 'BMXBMI': 28.5, 'name': 'TEST_ONLY'}),
    ('string_age', {'RIDAGEYR': '45', 'RIAGENDR': 1, 'BMXBMI': 28.5}),
]
invalid_results = []
for name, payload in invalid_cases:
    response = requests.post(PREDICT_URL, json=payload, timeout=10)
    invalid_results.append({'name': name, 'status_code': response.status_code})
    assert response.status_code == 422, f'{name}: expected 422, got {response.status_code}'
print('REJECTED:', len(invalid_results), '/', len(invalid_cases))
print(invalid_results)

REJECTED: 6 / 6
[{'name': 'age_out_of_range', 'status_code': 422}, {'name': 'invalid_gender', 'status_code': 422}, {'name': 'bmi_out_of_range', 'status_code': 422}, {'name': 'missing_bmi', 'status_code': 422}, {'name': 'extra_field', 'status_code': 422}, {'name': 'string_age', 'status_code': 422}]


## 5. Lưu manifest kiểm thử (không chứa payload đầu vào hoặc dữ liệu cá thể)

File chỉ lưu số ca, trạng thái, hash và cấu hình kỹ thuật. Chưa kết luận an toàn để dùng y tế.

In [12]:
report = {
    'experiment': 'EXP-DEPLOY-8B-001',
    'stage': 'phase3_step8b_local_fastapi_smoke_test',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'research_only_not_clinically_validated',
    'model_sha256': SHA,
    'locked_threshold': THRESHOLD,
    'synthetic_valid_cases_passed': len(valid_results),
    'invalid_cases_rejected': len(invalid_results),
    'health_live_ok': live.status_code == 200,
    'health_ready_ok': ready.status_code == 200,
    'fit_executed': False,
    'test_reopened': False,
    'model_modified': False,
    'api_publicly_exposed': False,
    'clinical_validation': False,
    'notes': 'Research-only local API; no actual health data sent; model unchanged.'
}
OUT.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUT / '08b_api_smoke_results.json'
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', REPORT_PATH)

Saved: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\outputs\model_deployment\fastapi_8b\08b_api_smoke_results.json


## 6. Output cuối gửi lại

Sau khi chạy xong các ô, gửi output cuối hoặc JSON tổng hợp cho tôi. Nếu lỗi, gửi lỗi từ PowerShell, không gửi mật khẩu/token/dữ liệu sức khỏe thật.

In [13]:
print('===== BƯỚC 8B — LOCAL FASTAPI SMOKE TEST (NGƯỜI DÙNG TỰ CHẠY) =====')
print('Model SHA256:', SHA)
print('Server ready:', report['health_ready_ok'])
print('Synthetic valid:', report['synthetic_valid_cases_passed'], '/ 2')
print('Rejected invalid:', report['invalid_cases_rejected'], '/ 6')
print('Locked threshold:', THRESHOLD)
print('Report:', REPORT_PATH)
print('TRAIN FIT: NO | TEST OPENED: NO | MODEL MODIFIED: NO | PUBLIC API: NO')
print('===== KẾT THÚC — GỬI OUTPUT CHO TÔI TRƯỚC BƯỚC 8C =====')

===== BƯỚC 8B — LOCAL FASTAPI SMOKE TEST (NGƯỜI DÙNG TỰ CHẠY) =====
Model SHA256: 31e9c94b0f14369be7920b9315b026d0c4afedb9f7b2885d120d631d930582b1
Server ready: True
Synthetic valid: 2 / 2
Rejected invalid: 6 / 6
Locked threshold: 0.225
Report: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\outputs\model_deployment\fastapi_8b\08b_api_smoke_results.json
TRAIN FIT: NO | TEST OPENED: NO | MODEL MODIFIED: NO | PUBLIC API: NO
===== KẾT THÚC — GỬI OUTPUT CHO TÔI TRƯỚC BƯỚC 8C =====
